## Install Dependencies

In [ ]:
!uv pip install syftbox-enclave syft-rds==0.1.1-dev.5

In [ ]:
from syft_rds import init_session
from syft_core import Client

In [ ]:
DATA_OWNERS = [
            "do-1-oc@openmined.org",
            "do-2-oc@openmined.org",
              ]
DATA_SCIENTIST = [ Client.load().email ]

In [ ]:
ds_clients = []

for do in DATA_OWNERS:
    ds_client = init_session(do)
    ds_clients.append(ds_client)
    print(f"Logged into {ds_client.host}")


In [ ]:
datasets = []

# We could either index the datasets
for ds_client in ds_clients:
    datasets.append(ds_client.datasets[0])
    
    

In [ ]:
datasets

### Inspect the data and inform their analysis



In [ ]:
# Inspect the mock data of all datasets

import pandas as pd
from pathlib import Path
from IPython.display import display

for dataset in datasets:
    mock_path = dataset.get_mock_path()
    mock_csv_path = list(Path(mock_path).glob("*.csv"))[0]
    df_mock = pd.read_csv(mock_csv_path)
    
    display(df_mock.head(3))
    print("\n\n\n")
    

### Propose analysis

In [ ]:

code_path = Path(".") / "code"
code_path.mkdir(exist_ok=True)

code_file_name  = "crop_analysis.py"
code_file_path = code_path / code_file_name

In [ ]:
%%writefile {code_file_path}

import os
from pathlib import Path
from sys import exit

import pandas as pd

DATA_DIR = os.environ["DATA_DIR"]
OUTPUT_DIR = os.environ["OUTPUT_DIR"]

dataset_paths = [ Path(dataset_path) for dataset_path in DATA_DIR.split(",")]
csv_paths = []
for dataset_path in dataset_paths:
    csv_paths.extend(list(Path(dataset_path).glob("*.csv")))

total_carrots = 0
total_tomatoes = 0

for csv_path in csv_paths:
    if not csv_path.exists():
        print(f"Warning: CSV path does not exist: {csv_path}")
        exit(1)
    df = pd.read_csv(csv_path)
    total_carrots += df[df["Product name"] == "Carrots"]["Quantity"].sum()
    total_tomatoes += df[df["Product name"] == "Tomatoes"]["Quantity"].sum()

print(f"Total Carrots: {total_carrots}\n")
print(f"Total Tomatoes: {total_tomatoes}\n")

with open(os.path.join(OUTPUT_DIR, "output.txt"), "w") as f:
    f.write(f"Total Carrots: {total_carrots}\n")
    f.write(f"Total Tomatoes: {total_tomatoes}\n")

### Test the analysis code against mock data

Before submitting the code for review, Treasury can test their analysis against the Department of Health mock data to ensure it works correctly.


In [ ]:
# Test against mock of all datasets

import subprocess, tempfile, os

with tempfile.TemporaryDirectory() as temp_dir:
    env = os.environ.copy()
    env.update({'DATA_DIR': ",".join([ str(dataset.get_mock_path()) for dataset in datasets]), 'OUTPUT_DIR': temp_dir})
    
    result = subprocess.run(["python", str(code_file_path)], env=env, capture_output=True, text=True)
    print(result.stdout)

In [ ]:
from uuid import uuid4

# Generate 
RANDOM_ID = str(uuid4())[0:8]
JOB_NAME = f"Test Job - {RANDOM_ID}"

print("Job Name:", JOB_NAME)

In [ ]:
ENCLAVE = "enclave-organic-coop@openmined.org"

jobs = []

for ds_client, dataset in zip(ds_clients, datasets):
    job = ds_client.jobs.submit(
                name=JOB_NAME,
                description="Organic Coop Experiment",
                user_code_path=code_path,
                dataset_name=dataset.name,
                entrypoint = code_file_name,
                enclave = ENCLAVE
            )
    job.describe()
    jobs.append(job)

In [ ]:
ds_clients[0].jobs.get(name=JOB_NAME).describe()

## Enclave Client

In [ ]:
from syftbox_enclave import connect

In [ ]:
enclave_client = connect(ENCLAVE)

In [ ]:
PROJECT_NAME = f"Test Project - {RANDOM_ID}"

print("Project Name:", PROJECT_NAME)

In [ ]:
datasets

In [ ]:

proj_res = enclave_client.create_project(
                   project_name = PROJECT_NAME,
                   datasets = datasets,
                   output_owners = DATA_OWNERS + DATA_SCIENTIST,
                   code_path = code_path,
                   entrypoint = code_file_name,
            )

In [ ]:
proj_res.status(block=True)

In [ ]:
# Force Start if atleast one of the datasites have approved
# proj_res.force_start()


# 5. Access Output Results

In [ ]:
# Wait until the project is ready
proj_res_path = proj_res.output(block=True)

In [ ]:
output_file_path = proj_res_path / "output.txt"

In [ ]:
with open(output_file_path ,"r") as f:
    print(f.read())
